---

# **[실습] 주택청약 FAQ 시스템 구현**

### **문제 설명**
이전 코드를 기반으로 주택청약 FAQ 시스템을 다음 요구사항에 맞춰 개선합니다. 

1. 응답 품질 향상 (1개 이상)
   - 생성된 답변의 품질을 평가 (답변이 불충분한 경우 예외 처리)
   - 관련성 높은 FAQ 문서 검색 (임베딩 모델, 청크 크기, 벡터 검색 방법 등)

2. 사용자 경험 개선 (1개 이상)
   - 대화 이력 관리 기능 추가 (요약, 트리밍 기능 등 고려)
   - 최근 대화 기반 컨텍스트 구성 
   - 사용자 프로필 기반 맞춤 응답

### **제약 조건**
- Gradio ChatInterface 사용
- RAG 구조 유지

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
import json
from pprint import pprint

## step 0) 벡터 저장소 준비

- PRJ01_W3_006에서 전처리된 JSON 파일을 사용
- OpenAI 임베딩으로 Chroma DB 구축 (최초 1회만 실행)

In [3]:
# 문서 로드
from langchain_core.documents import Document

output_file = ".././data/housing_faq_formatted.json"

with open(output_file, 'r', encoding='utf-8-sig') as f:
    formatted_docs = [Document(**doc) for doc in json.load(f)]

print(f"로드된 문서 수: {len(formatted_docs)}")
print(formatted_docs[0].page_content)

로드된 문서 수: 50
[1]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?
답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.



In [17]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문서 벡터 저장 (최초 1회)
vector_store = Chroma.from_documents(
    documents=formatted_docs,
    embedding=embeddings,
    collection_name="housing_faq_db_homework",
    persist_directory="./chroma_db",
)
print(f"저장된 문서 수: {vector_store._collection.count()}")

저장된 문서 수: 50


In [21]:
vector_store._collection.count()

50

In [ ]:
# vector_store.delete_collection()

## step 1) RAG 시스템 + 대화 이력 관리

### 개선 사항 (수업 2.2·2.4·3.1 통합)
- **MessagesPlaceholder** — `ChatPromptTemplate`에 이전 대화 삽입 (수업 1)
- **RunnableWithMessageHistory** — 히스토리 자동 저장·로드 (수업 2)
- **SummarizedSQLiteHistory** — SQLite 영구 저장 + 자동 요약 (수업 2.2 + 2.4)
- **create_agent + InMemorySaver** — LangChain 1.0 권장 패턴 (수업 3.1)
- **gr.State** — Gradio 사용자별 session_id 관리

In [26]:
import gradio as gr
import json
import uuid
import sqlite3
from typing import List, Optional, Generator, Literal
from dataclasses import dataclass

from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import BaseChatModel
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


# ── 데이터 클래스 ──────────────────────────────────────────

class AnswerQualityOutput(BaseModel):
    """LLM 기반 답변 품질 평가 결과"""
    is_sufficient: bool = Field(description="답변이 충분하면 True, 불충분이면 False")
    reason: str = Field(description="평가 이유 한 문장")


class MetadataFilter(BaseModel):
    """Chroma DB 메타데이터 필터 조건"""
    keyword: Optional[str] = Field(default=None)
    keyword_operator: Optional[Literal["$eq", "$ne"]] = Field(default=None)
    question_id_min: Optional[int] = Field(default=None)
    question_id_min_operator: Optional[Literal["$gt", "$gte"]] = Field(default=None)
    question_id_max: Optional[int] = Field(default=None)
    question_id_max_operator: Optional[Literal["$lt", "$lte"]] = Field(default=None)
    logical_operator: Optional[Literal["$and", "$or"]] = Field(default="$and")


@dataclass
class SearchResult:
    context: str
    source_documents: Optional[List]


# ── SummarizedSQLiteHistory (수업 2.2 SQLite + 2.4 요약 통합) ─

class SummarizedSQLiteHistory(BaseChatMessageHistory):
    """SQLite 영구 저장 + 자동 요약이 결합된 대화 이력 관리
    
    - 수업 2.2 SQLiteChatMessageHistory: SQLite 기반 영구 저장
    - 수업 2.4 SummarizedInMemoryHistory: 임계값 초과 시 LLM 요약
    """

    def __init__(self, session_id: str, db_path: str = "chat_history_rag.db",
                 summary_threshold: int = 6, llm=None):
        self.session_id = session_id
        self.db_path = db_path
        self.summary_threshold = summary_threshold
        self.llm = llm or ChatOpenAI(model="gpt-4.1-mini", temperature=0.1)
        self._create_table()

    def _create_table(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS messages (
                    id         INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    msg_type   TEXT NOT NULL,
                    content    TEXT NOT NULL,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """)

    @property
    def messages(self) -> List[BaseMessage]:
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT msg_type, content FROM messages WHERE session_id=? ORDER BY created_at",
                (self.session_id,)
            ).fetchall()
        type_map = {"HumanMessage": HumanMessage, "AIMessage": AIMessage}
        return [type_map.get(t, SystemMessage)(content=c) for t, c in rows]

    def add_messages(self, new_msgs: List[BaseMessage]) -> None:
        """새 메시지 DB 저장 후 임계값 초과 시 자동 요약"""
        with sqlite3.connect(self.db_path) as conn:
            conn.executemany(
                "INSERT INTO messages (session_id, msg_type, content) VALUES (?,?,?)",
                [(self.session_id, m.__class__.__name__, m.content) for m in new_msgs]
            )
        current = self.messages
        print(f"[이력] 현재 메시지 수: {len(current)}")
        if len(current) >= self.summary_threshold:
            self._summarize_and_compress(current)

    def _summarize_and_compress(self, msgs: List[BaseMessage]):
        """오래된 메시지를 LLM으로 요약 후 압축 (수업 2.4 참고)"""
        to_summarize, to_keep = msgs[:-2], msgs[-2:]
        if not to_summarize:
            return
        summary = self.llm.invoke([
            SystemMessage(content=(
                "Summarize the following conversation accurately. "
                "Preserve key facts and information. Use Korean."
            )),
            *to_summarize,
            HumanMessage(content="위 대화를 핵심 내용 위주로 간결하게 요약하세요.")
        ])
        print(f"→ 요약 완료 ({len(to_summarize)}개 → 1개)")
        compressed = [
            SystemMessage(content=f"[이전 대화 요약]\n{summary.content}"),
            *to_keep,
        ]
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("DELETE FROM messages WHERE session_id=?", (self.session_id,))
            conn.executemany(
                "INSERT INTO messages (session_id, msg_type, content) VALUES (?,?,?)",
                [(self.session_id, m.__class__.__name__, m.content) for m in compressed]
            )

    def clear(self) -> None:
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("DELETE FROM messages WHERE session_id=?", (self.session_id,))


# 세션 히스토리 팩토리
_history_cache: dict = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """session_id별 SummarizedSQLiteHistory 반환 (없으면 생성)"""
    if session_id not in _history_cache:
        _history_cache[session_id] = SummarizedSQLiteHistory(
            session_id=session_id, summary_threshold=6
        )
    return _history_cache[session_id]


print("✅ SummarizedSQLiteHistory 준비 완료")


✅ SummarizedSQLiteHistory 준비 완료


In [27]:
# ── RAGSystem (수업 1·2 통합: MessagesPlaceholder + RunnableWithMessageHistory) ─

class RAGSystem:
    def __init__(self, llm: BaseChatModel, eval_llm: BaseChatModel, retriever):
        self.llm = llm
        self.eval_llm = eval_llm
        if not retriever:
            raise ValueError("retriever가 필요합니다.")
        self.retriever = retriever
        self.vs = None
        self._build_chain()

    def _build_chain(self):
        """MessagesPlaceholder + RunnableWithMessageHistory 기반 RAG 체인 구성"""
        # 수업 1: MessagesPlaceholder로 이전 대화를 HumanMessage/AIMessage 객체로 삽입
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "다음 지침을 따라 답변하세요:\n"
             "1. 주어진 문서만을 기반으로 답변\n"
             "2. 근거 없는 내용은 '근거 없음'\n"
             "3. 모르면 '잘 모르겠습니다'\n"
             "4. 이전 대화 맥락을 참고해 연속 질문에 자연스럽게 답변\n\n"
             "[참고 문서]\n{context}"),
            MessagesPlaceholder(variable_name="history"),   # ← 수업 핵심
            ("human", "{question}"),
        ])
        # 수업 2: RunnableWithMessageHistory로 히스토리 자동 저장·로드
        self.chain_with_history = RunnableWithMessageHistory(
            prompt | self.llm | StrOutputParser(),
            get_session_history,
            input_messages_key="question",
            history_messages_key="history",
        )

    def update_retriever(self, retriever, vs=None):
        """설정 변경 시 retriever 교체 후 체인 재구성"""
        self.retriever = retriever
        self.vs = vs
        self._build_chain()

    # ── 포맷 헬퍼 ──────────────────────────────────────────

    def _format_docs(self, docs: List) -> str:
        return "\n\n".join(d.page_content for d in docs)

    def _format_source_documents(self, docs: Optional[List]) -> str:
        if not docs:
            return "\n\nℹ️ 관련 문서를 찾을 수 없습니다."
        parts = []
        for i, doc in enumerate(docs, 1):
            m = doc.metadata if hasattr(doc, "metadata") else {}
            info = []
            if "question_id" in m: info.append(f"ID: {m['question_id']}")
            if "keyword" in m:     info.append(f"키워드: {m['keyword']}")
            if "summary" in m:     info.append(f"요약: {m['summary']}")
            parts.append(
                f"📚 참조 문서 {i}\n"
                f"• {' | '.join(info) or '출처 없음'}\n"
                f"• 내용: {doc.page_content}"
            )
        return "\n\n" + "\n\n".join(parts)

    # ── 평가 헬퍼 ──────────────────────────────────────────

    def _evaluate_answer_quality(self, question: str, answer: str) -> AnswerQualityOutput:
        prompt = ChatPromptTemplate.from_messages([
            ("system", "답변이 질문에 충분히 답하는지 평가하세요.\n"
             "1. 핵심을 구체적으로 다루고 있는가?\n"
             "2. 불충분한 표현만 있지 않은가?\n"
             "3. 논리적으로 추론 가능한 답변인가?"),
            ("human", "질문: {question}\n\n답변: {answer}")
        ])
        return (prompt | self.eval_llm.with_structured_output(AnswerQualityOutput)).invoke(
            {"question": question, "answer": answer}
        )

    def _check_relevance(self, docs: List, question: str) -> List:
        if not docs:
            return []
        prompt = ChatPromptTemplate.from_messages([
            ("system", "컨텍스트가 질문 답변에 필요한 정보를 포함하는지 평가하세요.\n"
             "'Yes' 또는 'No'로만 답하세요."),
            ("human", "[컨텍스트]\n{context}\n\n[질문]\n{question}")
        ])
        chain = prompt | self.eval_llm | StrOutputParser()
        relevant = []
        for doc in docs:
            r = chain.invoke({"context": doc.page_content, "question": question}).lower()
            print(f"문서 {doc.metadata.get('question_id','?')} 관련성: {r}")
            if "yes" in r:
                relevant.append(doc)
        return relevant

    # ── RAG 핵심 메서드 ────────────────────────────────────

    def search_documents(self, question: str) -> SearchResult:
        try:
            if self.vs is not None and isinstance(self.vs, Chroma):
                try:
                    fp = metadata_extraction_chain.invoke({"query": question})
                    fd = build_chroma_filter(fp)
                    if fd:
                        print(f"[필터 적용] {fd}")
                        docs = self.vs.as_retriever(
                            search_kwargs={"filter": fd, "k": 4}
                        ).invoke(question)
                    else:
                        docs = self.retriever.invoke(question)
                except Exception as fe:
                    print(f"[필터 오류→기본] {fe}")
                    docs = self.retriever.invoke(question)
            else:
                docs = self.retriever.invoke(question)

            print(f"검색된 문서: {len(docs)}개")
            rel = self._check_relevance(docs, question)
            print(f"관련 문서: {len(rel)}개")
            return SearchResult(
                context=self._format_docs(rel) if rel else "관련 문서 없음",
                source_documents=rel,
            )
        except Exception as e:
            print(f"검색 오류: {e}")
            return SearchResult(context="검색 오류 발생", source_documents=None)

    def generate_answer(self, message: str, session_id: str) -> Generator[str, None, None]:
        """RunnableWithMessageHistory + MessagesPlaceholder 기반 스트리밍 답변"""
        sr = self.search_documents(message)
        if not sr.source_documents:
            yield "죄송합니다. 관련 문서를 찾을 수 없습니다. 다른 질문을 해주시겠습니까?"
            return

        # session_id → SQLite에서 이전 대화 이력 자동 로드
        config = {"configurable": {"session_id": session_id}}
        full_answer = ""
        try:
            for chunk in self.chain_with_history.stream(
                {"context": sr.context, "question": message}, config=config
            ):
                full_answer += chunk
                yield full_answer

            quality = self._evaluate_answer_quality(message, full_answer)
            sources = self._format_source_documents(sr.source_documents)

            if not quality.is_sufficient:
                print(f"[품질] 불충분: {quality.reason}")
                yield (
                    "제공된 FAQ 문서에서 직접적인 답변을 찾지 못했습니다.\n"
                    f"*(평가 이유: {quality.reason})*\n\n---\n{sources}"
                )
            else:
                print(f"[품질] 충분: {quality.reason}")
                yield f"{full_answer}\n\n---\n{sources}"
        except Exception as e:
            yield f"답변 생성 중 오류 발생: {e}"


print("✅ RAGSystem 준비 완료")


✅ RAGSystem 준비 완료


In [28]:
# ── 임베딩 & 벡터 저장소 빌더 ─────────────────────────────

def build_embedding_model(name: str):
    if name == "OpenAI text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "HuggingFace BAAI/bge-m3":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
    elif name == "Ollama bge-m3":
        from langchain_community.embeddings import OllamaEmbeddings
        return OllamaEmbeddings(model="bge-m3")
    return OpenAIEmbeddings(model="text-embedding-3-small")


def build_retriever(emb_name: str, vs_type: str, search_type: str,
                    top_k: int, threshold: float):
    emb = build_embedding_model(emb_name)
    if vs_type == "Chroma":
        vs = Chroma(collection_name="housing_faq_db_homework",
                    persist_directory="./chroma_db", embedding_function=emb)
    else:
        with open(".././data/housing_faq_formatted.json", "r", encoding="utf-8-sig") as f:
            docs = [Document(**d) for d in json.load(f)]
        vs = FAISS.from_documents(docs, emb)

    if threshold > 0:
        ret = vs.as_retriever(search_type="similarity_score_threshold",
                               search_kwargs={"k": top_k, "score_threshold": threshold})
    elif search_type == "mmr":
        ret = vs.as_retriever(search_type="mmr",
                               search_kwargs={"k": top_k, "fetch_k": top_k * 3, "lambda_mult": 0.5})
    else:
        ret = vs.as_retriever(search_type="similarity", search_kwargs={"k": top_k})
    return ret, vs


def build_chroma_filter(fp: MetadataFilter) -> dict:
    conds = []
    if fp.keyword and fp.keyword_operator:
        conds.append({"keyword": {fp.keyword_operator: fp.keyword}})
    if fp.question_id_min is not None and fp.question_id_min_operator:
        conds.append({"question_id": {fp.question_id_min_operator: fp.question_id_min}})
    if fp.question_id_max is not None and fp.question_id_max_operator:
        conds.append({"question_id": {fp.question_id_max_operator: fp.question_id_max}})
    if not conds:
        return {}
    return conds[0] if len(conds) == 1 else {fp.logical_operator or "$and": conds}


# ── RAGSystem 초기화 ────────────────────────────────────────

llm      = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
eval_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

default_retriever, default_vs = build_retriever(
    "OpenAI text-embedding-3-small", "Chroma", "similarity", 3, 0.0
)
rag_system     = RAGSystem(llm=llm, eval_llm=eval_llm, retriever=default_retriever)
rag_system.vs  = default_vs

# 메타데이터 추출 체인
actual_keywords = sorted(set(doc.metadata["keyword"] for doc in formatted_docs))
METADATA_SYSTEM = (
    f"사용자 쿼리에서 Chroma DB 필터 조건을 추출합니다.\n"
    f"키워드는 반드시 다음 목록에서만 선택: {actual_keywords}\n"
    "해당 없으면 null 반환. 질문 ID 범위: 'N번 이상'→$gte, 'N번 이하'→$lte."
)
metadata_extraction_chain = (
    ChatPromptTemplate.from_messages([
        ("system", METADATA_SYSTEM),
        ("human", "{query}")
    ])
    | ChatOpenAI(model="gpt-4.1-mini", temperature=0).with_structured_output(MetadataFilter)
)
print("✅ RAGSystem + 메타데이터 체인 초기화 완료")


✅ RAGSystem + 메타데이터 체인 초기화 완료


In [29]:
# ── create_agent + InMemorySaver (수업 섹션 3.1 — LangChain 1.0 권장 패턴) ──

@tool
def search_housing_faq(query: str) -> str:
    """주택청약 FAQ 문서에서 질문 관련 정보를 검색합니다."""
    result = rag_system.search_documents(query)
    return result.context if result.context else "관련 문서를 찾을 수 없습니다."


# InMemorySaver: 메모리 기반 checkpointer (수업 3.1)
_agent_checkpointer = InMemorySaver()

agent = create_agent(
    model="gpt-4.1-nano",
    tools=[search_housing_faq],
    system_prompt=(
        "당신은 주택청약 FAQ 전문 어시스턴트입니다. "
        "반드시 search_housing_faq 도구로 문서를 검색한 후 답변하세요. "
        "문서에 없는 내용은 '문서에 근거 없음'이라고 답변하세요."
    ),
    checkpointer=_agent_checkpointer,
)
print("✅ create_agent + InMemorySaver 준비 완료")

# ── 동작 확인 예시 (주석 해제 후 실행) ──────────────────────
# config = {"configurable": {"thread_id": "test_1"}}
# response = agent.invoke(
#     {"messages": [{"role": "user", "content": "무주택 세대에 대해 설명해주세요."}]},
#     config=config
# )
# print(response["messages"][-1].content)


✅ create_agent + InMemorySaver 준비 완료


In [31]:
# ── Gradio UI — gr.ChatInterface + gr.State(session_id) + SQLite 이력 연동 ──

_last_settings: dict = {}

def chat_with_settings(message, history, session_id,
                        embedding_model, vector_store_type, search_type, top_k, threshold):
    """설정 변경 감지 → retriever 재생성 후 RAG 답변 스트리밍"""
    global _last_settings
    cur = {
        "emb": embedding_model, "vs": vector_store_type,
        "search": search_type, "k": top_k, "thr": threshold
    }
    if cur != _last_settings:
        ret, vs = build_retriever(
            embedding_model, vector_store_type, search_type, int(top_k), threshold
        )
        rag_system.update_retriever(ret, vs)
        _last_settings = cur
    yield from rag_system.generate_answer(message, session_id)


demo = gr.ChatInterface(
    fn=chat_with_settings,
    title="🏠 주택청약 FAQ 챗봇",
    description=(
        "💾 대화 이력은 SQLite(chat_history_rag.db)에 영구 저장됩니다. "
        "🗜️ 메시지가 6개를 초과하면 LLM이 자동으로 이전 대화를 요약합니다."
    ),
    additional_inputs=[
        # gr.State: 사용자 탭마다 고유 UUID 생성 → SQLite 이력과 1:1 매핑
        gr.State(value=lambda: str(uuid.uuid4())),
        gr.Dropdown(
            choices=["OpenAI text-embedding-3-small", "HuggingFace BAAI/bge-m3", "Ollama bge-m3"],
            value="OpenAI text-embedding-3-small",
            label="🤖 임베딩 모델",
        ),
        gr.Dropdown(
            choices=["Chroma", "FAISS"],
            value="Chroma",
            label="🗄️ 벡터 저장소",
        ),
        gr.Dropdown(
            choices=["similarity", "mmr"],
            value="similarity",
            label="🔍 검색 방식",
        ),
        gr.Slider(minimum=1, maximum=10, value=3, step=1,       label="📦 Top-K"),
        gr.Slider(minimum=0.0, maximum=1.0, value=0.0, step=0.05, label="📏 유사도 임계값"),
    ],
    additional_inputs_accordion=gr.Accordion(label="⚙️ RAG 설정", open=False),
    examples=[
        ["수원시의 주택건설지역은 어디에 해당하나요?"],
        ["무주택 세대에 대해서 설명해주세요."],
        ["청약통장 납입 인정 금액은 얼마인가요?"],
    ],
)
demo.launch()


Running on local URL:  http://127.0.0.1:7864

To create a public link, set `share=True` in `launch()`.


In [32]:
demo.close()

Closing server running on port: 7864
